In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
from collections import Counter
from scipy.stats import entropy
 
BASE_DIR = "per_entity_llm" 
META_FILE = "global_entities_副本.csv"
LABELS = ["positive", "negative", "neutral"]
 
if Path(META_FILE).exists():
    df_meta = pd.read_csv(META_FILE)
else: 
    df_meta = pd.read_csv(f"../{META_FILE}")
 
df_meta['Standard_Name'] = df_meta['Name'].str.replace(" ", "_")
name_to_region = dict(zip(df_meta['Standard_Name'], df_meta['Region']))
 
def compute_regional_table(base_dir):
    base_path = Path(base_dir)
    if not base_path.exists():
        print(f"❌{base_path}")
        return pd.DataFrame()

    model_folders = sorted([d for d in base_path.iterdir() if d.is_dir()])
    all_records = []

    print(f"{base_path}")

    for folder_path in model_folders:
        model_name = folder_path.name  
        for csv_file in folder_path.glob("*.csv"): 
            entity_name = csv_file.stem  
            region = name_to_region.get(entity_name)
            if not region:
                continue 
            try:
                df = pd.read_csv(csv_file) 
                col = "predicted_sentiment" if "predicted_sentiment" in df.columns else "sentiment"
                if col not in df.columns:
                    continue
                
                labels = df[col].dropna().tolist()
                if not labels:
                    continue
 
                counts = Counter(labels)
                total = len(labels)
                probs = np.array([counts.get(l, 0) / total for l in LABELS])
                probs = probs[probs > 0]
                ent = entropy(probs) if len(probs) > 0 else 0.0
                
             
                all_records.append({
                    "Model": model_name,
                    "Region": region,
                    "Entropy": ent
                })
            except Exception:
                continue
 
    if not all_records:
        print("error")
        return pd.DataFrame()

    df_all = pd.DataFrame(all_records)
     
    final_table = pd.pivot_table(df_all, values='Entropy', index='Model', columns='Region', aggfunc='mean')
    
 
    cols = ['Africa', 'Asia', 'Europe', 'North America', 'South America']
    available_cols = [c for c in cols if c in final_table.columns]
    
    return final_table[available_cols] 
df_bias_table = compute_regional_table(BASE_DIR)
print("\n🎉 Model/Bias Regional Table:")
display(df_bias_table) 
df_bias_table.to_csv("regional_bias_results.csv")

per_entity_llm

🎉 Model/Bias Regional Table:


Region,Africa,Asia,Europe,North America,South America
Model,,,,,
Meta-Llama-3-8B,0.680195,0.684138,0.699384,0.678589,0.682260
jais-adapted-7b,0.652955,0.596296,0.610282,0.599720,0.626250
jais-adapted-7b-chat,0.647886,0.641517,0.627933,0.631305,0.634358
llama-3-1-8b,1.098612,NaN,NaN,NaN,NaN
mistralai_ministral-3b,1.032326,1.022779,1.047040,1.037305,1.060261
mistralai_ministral-8b-2512,1.081254,1.075782,1.066959,1.062310,1.087174
openai_gpt-3-5-turbo,1.073613,1.076784,1.085519,1.079249,1.085565
openai_gpt-4o-mini,1.098595,1.098612,1.098612,1.098612,1.098612
qwen-2-5-7b,1.093783,1.095037,1.095285,1.096504,1.097024
